<a href="https://colab.research.google.com/github/addadugurudurga2024-lang/Flyrank/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

I am using the Refresh / Content Opportunity Scoring lane.

One row represents one content item for one reporting date in the
`fact_content_daily_performance` table.

For this assignment, I use the middle month `2026-03` for development
and verification. I use this month instead of the `_sample` table because
the `_sample` table represents the final month and should be treated as a
sealed outcome period.

The decision-support goal is to identify content items that show signals
that may make them suitable for refresh review.

In [8]:
import duckdb

con = duckdb.connect()

# Connect DuckDB to the Hugging Face warehouse
con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_daily_sample": f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

print("DuckDB connected to FlyRank warehouse.")

for name, src in TABLES.items():
    n = con.sql(f"SELECT COUNT(*) FROM {src}").fetchone()[0]
    print(f"{name:22} {n:>12,} rows")

DuckDB connected to FlyRank warehouse.
dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


## 2. Fields: feature / label / context / excluded

### Features

I will use observable signals that are available at the decision moment,
such as impressions, clicks, sessions, average position, content age,
and freshness-related information.

### Label / proxy

For this assignment, the target is a future content-performance outcome
that can be defined from a later observation window. A page that shows
declining performance in the future window can be treated as a decline
proxy for refresh review.

### Context

Client and content identifiers are used only for grouping, joining,
and checking the grain. They are not used as predictive features.

### Excluded

I deliberately exclude future-window measurements and any field derived
from the future outcome from the feature set because those values would
not be known at the decision moment. I also exclude identifier fields
from modeling because the pseudonymous IDs have no meaningful predictive
value.

## 3. Verify it with queries (grain, counts, missing values, windows)
### Query 1 — Grain check

I check whether the combination of report date, client, and content uniquely
identifies a row in the March 2026 slice.

In [15]:
# Query 1 — Verify the grain for March 2026

query = f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS row_count
FROM {TABLES['fact_daily']}
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
ORDER BY row_count DESC
LIMIT 10
"""

result = con.sql(query).df()

print("Duplicate grain combinations found:", len(result))
display(result)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate grain combinations found: 0


,report_date,client_hash_id,content_hash_id,row_count


### Query 2 — Row count and date span

I measure the number of rows in the March 2026 development slice and verify
the earliest and latest reporting dates present in the slice.

In [10]:
# Query 2 — Row count and date span for March 2026

query_2 = f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM {TABLES['fact_daily']}
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
"""

date_check = con.sql(query_2).df()

display(date_check)

,row_count,first_date,last_date
0,9841378,2026-03-01,2026-03-31


### Query 3 — Data availability

I check how many March 2026 rows have GSC data explicitly available.
I use `IS TRUE` because the availability field can contain TRUE, FALSE,
or NULL values.

In [13]:
# Query 3 — GSC availability for March 2026

query_3 = f"""
SELECT
    COUNT(*) AS available_rows
FROM {TABLES['fact_daily']}
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
  AND gsc_data_available IS TRUE
"""

availability_check = con.sql(query_3).df()

display(availability_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,available_rows
0,3611061


## Five features

I use five observable features for the refresh-review decision:

1. `impressions` — measures how much search visibility the content receives.
   Available when? It is historical performance already recorded before
   the decision moment.

2. `clicks` — measures search clicks received by the content.
   Available when? It comes from the historical performance window available
   before the decision.

3. `sessions` — measures historical engagement/traffic activity.
   Available when? It is recorded before the decision moment and therefore
   does not require future information.

4. `avg_position` — represents the content's historical average search position.
   Available when? It is calculated from search performance already observed
   before the decision.

5. `content_age_days` — measures how old the content is.
   Available when? The content creation date is known before the decision
   moment.

## 4. Data limits

This data has an unbalanced history, so different clients do not necessarily
have the same amount of historical data.

Some early rows may have GSC search data without GA4 data. Therefore, missing
GA4 measurements cannot automatically be interpreted as zero engagement.

Another limitation is that the daily warehouse is time-based, so feature and
outcome windows must be kept separate. If future-window measurements are used
as features, the result can suffer from data leakage.

The analysis is therefore decision-support rather than proof that refreshing
a page will cause its performance to improve.

## Self-check

Before you submit, confirm each line honestly:

- [✅  ] Every section above is filled — markdown thinking AND the code that backs it
- [✅  ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅  ] No client names, URLs, or private queries anywhere
- [✅  ] My claims use careful words: observed, measured, directional, decision-support
- [✅  ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.